GRU

In [ ]:
!pip install gensim
!pip install tensorflow
from google.colab import files
uploaded=files.upload()


Saving test.csv to test (1).csv


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, GRU
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten, Input, Dropout
from sklearn.model_selection import train_test_split
from gensim.parsing.preprocessing import preprocess_string
from nltk import sent_tokenize

In [ ]:
df = pd.read_csv('data_mr.csv')

In [ ]:
df.drop_duplicates(inplace=True)


In [ ]:
df

,text,label
0,I love sci-fi and am willing to put up with a ...,0
1,"Worth the entertainment value of a rental, esp...",0
2,its a totally average film with a few semi-alr...,0
3,STAR RATING: ***** Saturday Night **** Friday ...,0
4,"First off let me say, If you haven't enjoyed a...",0
...,...,...
24995,Just got around to seeing Monster Man yesterda...,1
24996,I got this as part of a competition prize. I w...,1
24997,I got Monster Man in a box set of three films ...,1
24998,"Five minutes in, i started to feel how naff th...",1


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)    # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
y=df['label']

In [ ]:
X=df['text']

In [ ]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=7)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_words = 10000


In [ ]:
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
max_len = 300

X_train_pad = pad_sequences(X_train_seq,
                            maxlen=max_len,
                            padding='post')

X_test_pad = pad_sequences(X_test_seq,
                           maxlen=max_len,
                           padding='post')

In [ ]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    GRU(16, dropout=0.3),
    Dropout(0.3),
    Dense(1, activation='sigmoid', kernel_regularizer=l2(0.001))
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.5037 - loss: 0.6950 - val_accuracy: 0.5249 - val_loss: 0.6915
Epoch 2/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.5486 - loss: 0.6805 - val_accuracy: 0.5265 - val_loss: 0.6853
Epoch 3/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5704 - loss: 0.6384 - val_accuracy: 0.5350 - val_loss: 0.6696
Epoch 4/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.5839 - loss: 0.6028 - val_accuracy: 0.5522 - val_loss: 0.6846
Epoch 5/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5897 - loss: 0.5871 - val_accuracy: 0.5350 - val_loss: 0.6973
Epoch 6/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5987 - loss: 0.5795 - val_accuracy: 0.5431 - val_loss: 0.7117
Epoch 7/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.6457 - loss: 0.5546 - val_accuracy: 0.7898 - val_loss: 0.5179
Epoch 8/10
496/496 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.8682 - loss: 0.3278 - val_acc

In [ ]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.8711953235234832
F1 Score: 0.8721744348869774
